# Modelo de Espacio Vectorial

Imagina que tienes varios textos. La idea de Gerard Salton es convertir cada texto en una flecha (vector). 
Si dos textos hablan de lo mismo, sus flechas apuntan hacia la misma dirección. Si hablan de cosas distintas, sus flechas apuntan a lados diferentes.

---
## 1. Frases de Ejemplo
Usaremos 6 frases cortas divididas en tres temas: Minería física, Ciencia de Datos y una mascota (para comparar):

In [ ]:
import numpy as np
import pandas as pd

frases = [
    "el minero busca oro en la mina",         # Texto 1 (Minería física)
    "el minero extrae cobre de la mina",      # Texto 2 (Minería física)
    "el analista procesa datos en python",     # Texto 3 (Ciencia de datos)
    "el analista limpia datos en sql",          # Texto 4 (Ciencia de datos)
    "la mineria de datos descubre patrones",    # Texto 5 (Mezcla de palabras)
    "el perro corre en el parque"              # Texto 6 (Mascotas)
]

tokens = [f.split() for f in frases]
vocabulario = sorted(list(set(p for f in tokens for p in f)))

print(f"Total de frases: {len(frases)}")
print(f"Vocabulario ({len(vocabulario)} palabras): {vocabulario}")

Total de frases: 6
Vocabulario (22 palabras): ['analista', 'busca', 'cobre', 'corre', 'datos', 'de', 'descubre', 'el', 'en', 'extrae', 'la', 'limpia', 'mina', 'mineria', 'minero', 'oro', 'parque', 'patrones', 'perro', 'procesa', 'python', 'sql']


---
## 2. Convertir Frases a Números (TF-IDF)
* Palabras raras como **"perro"**, **"python"** o **"sql"** valen mucho porque dan información específica.
* Palabras comunes como **"el"**, **"en"** o **"datos"** valen menos porque aparecen en varios textos.

In [ ]:
N = len(frases)
V = len(vocabulario)

# Contar frecuencias (TF)
tf = np.zeros((N, V))
for i, t in enumerate(tokens):
    for p in t:
        tf[i, vocabulario.index(p)] += 1

# Calcular importancia (IDF)
df = (tf > 0).sum(axis=0)
idf = np.log2(N / df) + 1
tfidf = tf * idf

# Matriz TF-IDF
tabla_tfidf = pd.DataFrame(tfidf, index=[f"Texto {i+1}" for i in range(N)], columns=vocabulario)
tabla_tfidf.round(2)


         analista  busca  cobre  corre  datos    de  descubre    el    en  extrae   la  limpia  mina  mineria  minero   oro  parque  patrones  perro  procesa  python   sql
Texto 1      0.00   3.58   0.00   0.00    0.0  0.00      0.00  1.26  1.58    0.00  2.0    0.00  2.58     0.00    2.58  3.58    0.00      0.00   0.00     0.00    0.00  0.00
Texto 2      0.00   0.00   3.58   0.00    0.0  2.58      0.00  1.26  0.00    3.58  2.0    0.00  2.58     0.00    2.58  0.00    0.00      0.00   0.00     0.00    0.00  0.00
Texto 3      2.58   0.00   0.00   0.00    2.0  0.00      0.00  1.26  1.58    0.00  0.0    0.00  0.00     0.00    0.00  0.00    0.00      0.00   0.00     3.58    3.58  0.00
Texto 4      2.58   0.00   0.00   0.00    2.0  0.00      0.00  1.26  1.58    0.00  0.0    3.58  0.00     0.00    0.00  0.00    0.00      0.00   0.00     0.00    0.00  3.58
Texto 5      0.00   0.00   0.00   0.00    2.0  2.58      3.58  0.00  0.00    0.00  2.0    0.00  0.00     3.58    0.00  0.00    0.00      3.

---
## 3. ¿Qué tanto se parecen las frases? (Parecido y Centroide)
Calculamos la similaridad (Coseno) entre frases y el **Centroide ($C^*$)**, que es el promedio de todos nuestros textos.

In [ ]:
def coseno(u, v):
    norma_u, norma_v = np.linalg.norm(u), np.linalg.norm(v)
    return np.dot(u, v) / (norma_u * norma_v) if norma_u and norma_v else 0.0

sim = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        sim[i, j] = coseno(tfidf[i], tfidf[j])

centroide = tfidf.mean(axis=0)

tabla_sim = pd.DataFrame(sim, index=[f"Texto {i+1}" for i in range(N)], columns=[f"Texto {i+1}" for i in range(N)])
tabla_sim.round(2)


         Texto 1  Texto 2  Texto 3  Texto 4  Texto 5  Texto 6
Texto 1     1.00     0.39     0.09     0.09     0.08     0.12
Texto 2     0.39     1.00     0.03     0.03     0.20     0.06
Texto 3     0.09     0.03     1.00     0.37     0.09     0.13
Texto 4     0.09     0.03     0.37     1.00     0.09     0.13
Texto 5     0.08     0.20     0.09     0.09     1.00     0.00
Texto 6     0.12     0.06     0.13     0.13     0.00     1.00


---
## 4. ¿Qué tan amontonado está el grupo? (Densidad Q)
Sumamos el parecido de cada texto con el centroide promedio. A esa suma Salton le llamó **$Q$ (Densidad)**.

In [ ]:
parecidos = [coseno(centroide, tfidf[i]) for i in range(N)]
Q = sum(parecidos)
print(f"Densidad general del grupo (Q) = {Q:.3f}")

Densidad general del grupo (Q) = 3.131


---
## 5. El Experimento: ¿La palabra Ayuda o Estorba?
Hacemos un experimento muy simple para ver si una palabra ayuda a separar temas o solo hace bulto:
1. **Quitamos esa palabra** de los textos.
2. Medimos si el resto de los textos **se junta más** o **se separa más**.

### Regla fácil:
* Si al quitar la palabra los textos **se juntan más** $\rightarrow$ la palabra **estaba ayudando a separar** los temas (Ayuda a separar).
* Si al quitar la palabra los textos **se separan más** $\rightarrow$ la palabra **solo hacía bulto** y generaba ruido ( Hace bulto / estorba). Ejemplo: `"datos"`, `"el"`, `"en"`.

In [ ]:
resultados = []

for k, palabra in enumerate(vocabulario):
    # Simulamos quitar la palabra k
    sin_k = tfidf.copy()
    sin_k[:, k] = 0.0
    
    c_k = sin_k.mean(axis=0)
    Q_k = sum(coseno(c_k, sin_k[i]) for i in range(N))
    
    diferencia = Q_k - Q
    
    resultados.append({
        "Palabra": palabra,
        "Frecuencia (DF)": df[k],
        "¿Qué hace en el grupo?": "Ayuda a separar temas" if diferencia > 0 else "Hace bulto / estorba"
    })

pd.DataFrame(resultados)

,Palabra,Frecuencia (DF),¿Qué hace en el grupo?
0,busca,2,Hace bulto / estorba
1,corre,1,Ayuda a separar temas
2,datos,2,Hace bulto / estorba
3,de,1,Ayuda a separar temas
4,el,3,Hace bulto / estorba
5,en,1,Ayuda a separar temas
6,extrae,1,Ayuda a separar temas
7,la,1,Ayuda a separar temas
8,mineria,1,Ayuda a separar temas
9,minero,2,Hace bulto / estorba


---
## 6. La Solución de Salton: Transformar en Frases
En la tabla anterior la palabra `"datos"` sale marcando **Hace bulto / estorba** porque aparece en varias frases distintas.

Salton propone : Juntar palabras compuestas en una sola frase 
En lugar de dejar la palabra suelta `"datos"`, en el Texto 5 juntamos `"mineria de datos"` como una sola palabra compuesta: **`"mineria_datos"`**.


In [ ]:
# Nuevo corpus reemplazando "mineria de datos" por la frase compuesta "mineria_datos"
frases_frase = [
    "el minero busca oro en la mina",
    "el minero extrae cobre de la mina",
    "el analista procesa datos en python",
    "el analista limpia datos en sql",
    "la mineria_datos descubre patrones",  
    "el perro le va a isw fc"
]

tokens_f = [f.split() for f in frases_frase]
vocab_f = sorted(list(set(p for f in tokens_f for p in f)))
N_f, V_f = len(frases_frase), len(vocab_f)

tf_f = np.zeros((N_f, V_f))
for i, t in enumerate(tokens_f):
    for p in t:
        tf_f[i, vocab_f.index(p)] += 1

df_f = (tf_f > 0).sum(axis=0)
idf_f = np.log2(N_f / df_f) + 1
tfidf_f = tf_f * idf_f

c_f = tfidf_f.mean(axis=0)
Q_f = sum(coseno(c_f, tfidf_f[i]) for i in range(N_f))

# Calculamos el comportamiento de la frase "mineria_datos"
idx = vocab_f.index("mineria_datos")
sin_frase = tfidf_f.copy()
sin_frase[:, idx] = 0.0
c_sin = sin_frase.mean(axis=0)
Q_sin = sum(coseno(c_sin, sin_frase[i]) for i in range(N_f))
diff_frase = Q_sin - Q_f

print("=== RESULTADO DE LA SOLUCIÓN DE SALTON ===")
print(f"Palabra suelta 'datos':  Hace bulto / estorba")
print(f"Frase 'mineria_datos': {'Ayuda a separar temas' if diff_frase > 0 else 'Hace bulto'}")

=== RESULTADO DE LA SOLUCIÓN DE SALTON ===
Palabra suelta 'datos':  Hace bulto / estorba
Frase 'mineria_datos': Ayuda a separar temas
